# 02 – Aufbereitung der kleinräumigen Statistik

Ziel: alle 21 kleinräumigen Datensätze in **eine** einheitliche Tabelle im langen Format bringen, mit einer Zeile pro Gebiet, Merkmal und Zeitpunkt:

| gebiet | ebene | indikator | merkmal | zeitpunkt | jahr | wert | zeichen | hinweis |
|---|---|---|---|---|---|---|---|---|
| Zentrum | Ortsteil | einwohner | Einwohner insgesamt | 2024 | 2024 | 1893.0 | | |
| Zentrum | Ortsteil | sgb2 | Erwerbsfähige Leistungsberechtigte: Alleinerziehende | 2025 | 2025 | NaN | . | unbekannt oder aus Datenschutzgründen gesperrt |

Was dabei passiert und warum (Details im Entscheidungsprotokoll):

1. **Sonderzeichen** nach der Zeichenerklärung des Amts umsetzen: `-` → 0, `.` und `x` → fehlend. Das Originalzeichen bleibt in der Spalte `zeichen` (Nr. 13).
2. **Zahlen** von deutschem Format (`6,95`) in echte Zahlen umwandeln.
3. **Merkmalsnamen bereinigen:** Parteinamen vereinheitlichen (Nr. 12), doppelte Namen in SGB II eindeutig machen, Tippfehler korrigieren.
4. **Gebiete** mit Ebene (Ortsteil / Stadtbezirk / Stadt) und Ortsteilnummer versehen. Verknüpfung über den Namen (Nr. 9).
5. **Prüfen**, dass dabei nichts verloren geht.

Ergebnis: `data/processed/kleinraeumig_lang.csv` und `data/processed/indikatoren.csv`.

In [1]:
import re
from pathlib import Path

import geopandas as gpd
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

RAW = Path("data/raw")
PROCESSED = Path("data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

## 1. Die Indikatoren beschreiben

Zu jedem Datensatz halten wir fest, was er ist: Titel, Thema, ob die Zeitpunkte Jahre oder Wahltermine sind, und ob er auf der Bürgerumfrage beruht. Die Umfragedaten liegen nur alle zwei Jahre vor und schwanken stärker (Nr. 14).

Die Namen entsprechen den Dateinamen aus `download_data.py`.

In [2]:
INDIKATOREN = pd.DataFrame(
    [
        # indikator,              titel,                                   thema,             zeitart, umfrage
        ("einwohner",             "Einwohner (Hauptwohnsitz)",              "Bevölkerung",     "jahr", False),
        ("einwohner_alter",       "Einwohner nach Alter",                   "Bevölkerung",     "jahr", False),
        ("migrationshintergrund", "Einwohner mit Migrationshintergrund",    "Bevölkerung",     "jahr", False),
        ("einwohnerdichte",       "Einwohnerdichte",                        "Bevölkerung",     "jahr", False),
        ("haushalte",             "Personenhaushalte",                      "Bevölkerung",     "jahr", False),
        ("familien",              "Familien mit Kindern",                   "Bevölkerung",     "jahr", False),
        ("geborene_gestorbene",   "Geborene und Gestorbene",                "Bevölkerung",     "jahr", False),
        ("wanderungen",           "Wanderungen",                            "Bevölkerung",     "jahr", False),
        ("sgb2",                  "Grundsicherung für Arbeitssuchende (SGB II)", "Soziale Lage", "jahr", False),
        ("arbeitslose",           "Arbeitslose",                            "Soziale Lage",    "jahr", False),
        ("nettoeinkommen",        "Nettoeinkommen",                         "Soziale Lage",    "jahr", True),
        ("zufriedenheit",         "Zufriedenheitsfaktoren",                 "Soziale Lage",    "jahr", True),
        ("schulabschluesse",      "Schul- und Berufsabschlüsse",            "Bildung",         "jahr", True),
        ("mieten",                "Wohnungsmieten",                         "Wohnen",          "jahr", True),
        ("wohnsituation",         "Wohnsituation",                          "Wohnen",          "jahr", True),
        ("kfz_bestand",           "Kraftfahrzeugbestand",                   "Infrastruktur",   "jahr", False),
        ("flaechennutzung",       "Flächennutzung",                         "Infrastruktur",   "jahr", False),
        ("straftaten",            "Straftaten",                             "Sicherheit",      "jahr", False),
        ("bundestagswahlen",      "Bundestagswahlen",                       "Wahlen",          "wahl", False),
        ("landtagswahlen",        "Landtagswahlen",                         "Wahlen",          "wahl", False),
        ("stadtratswahlen",       "Stadtratswahlen",                        "Wahlen",          "wahl", False),
    ],
    columns=["indikator", "titel", "thema", "zeitart", "umfrage"],
)

# Passt die Liste zu den Dateien im Ordner?
dateien = {p.stem for p in (RAW / "kleinraeumig").glob("*.csv")}
assert dateien == set(INDIKATOREN["indikator"]), dateien ^ set(INDIKATOREN["indikator"])
INDIKATOREN

,indikator,titel,thema,zeitart,umfrage
0,einwohner,Einwohner (Hauptwohnsitz),Bevölkerung,jahr,False
1,einwohner_alter,Einwohner nach Alter,Bevölkerung,jahr,False
2,migrationshintergrund,Einwohner mit Migrationshintergrund,Bevölkerung,jahr,False
3,einwohnerdichte,Einwohnerdichte,Bevölkerung,jahr,False
4,haushalte,Personenhaushalte,Bevölkerung,jahr,False
5,familien,Familien mit Kindern,Bevölkerung,jahr,False
6,geborene_gestorbene,Geborene und Gestorbene,Bevölkerung,jahr,False
7,wanderungen,Wanderungen,Bevölkerung,jahr,False
8,sgb2,Grundsicherung für Arbeitssuchende (SGB II),Soziale Lage,jahr,False
9,arbeitslose,Arbeitslose,Soziale Lage,jahr,False


## 2. Gebiete: Ortsteile, Stadtbezirke, Stadt

Jede Statistik-CSV enthält 74 Gebiete: 63 Ortsteile, 10 Stadtbezirke und „Stadt Leipzig“. Die Geodaten liefern uns die amtliche Nummer und den Stadtbezirk jedes Ortsteils.

In [3]:
ortsteile = gpd.read_file(RAW / "geodaten/ortsteile.geojson")
bezirke = gpd.read_file(RAW / "geodaten/stadtbezirke.geojson")

gebiete = pd.concat([
    pd.DataFrame({
        "gebiet": ortsteile["Name"],
        "ebene": "Ortsteil",
        "nummer": ortsteile["OT"],
        "stadtbezirk": ortsteile["OT"].str[0].map(bezirke.set_index("SBZ")["Name"]),
    }),
    pd.DataFrame({
        "gebiet": bezirke["Name"],
        "ebene": "Stadtbezirk",
        "nummer": bezirke["SBZ"],
        "stadtbezirk": bezirke["Name"],
    }),
    pd.DataFrame({"gebiet": ["Stadt Leipzig"], "ebene": ["Stadt"], "nummer": [None], "stadtbezirk": [None]}),
], ignore_index=True)

assert gebiete["gebiet"].is_unique
gebiete["ebene"].value_counts()

ebene
Ortsteil       63
Stadtbezirk    10
Stadt           1
Name: count, dtype: int64

## 3. Einlesen und ins lange Format bringen

Die Rohdateien sind „breit“: eine Spalte pro Jahr bzw. Wahltermin. `melt` macht daraus eine Zeile pro Zeitpunkt.

Alles wird zunächst als **Text** gelesen. So sehen wir die Sonderzeichen, bevor sie zu Zahlen werden.

In [4]:
def lade_breit(indikator):
    df = pd.read_csv(RAW / "kleinraeumig" / f"{indikator}.csv", dtype=str)
    df = df.apply(lambda spalte: spalte.str.strip())
    zeitspalten = [c for c in df.columns if re.match(r"\d", c)]
    andere = set(df.columns) - set(zeitspalten) - {"Gebiet", "Sachmerkmal"}
    assert not andere, f"{indikator}: unerwartete Spalten {andere}"
    return df, zeitspalten


roh = {}
for indikator in INDIKATOREN["indikator"]:
    df, zeitspalten = lade_breit(indikator)
    lang = df.melt(id_vars=["Gebiet", "Sachmerkmal"], value_vars=zeitspalten, var_name="zeitpunkt", value_name="roh")
    lang.insert(0, "indikator", indikator)
    roh[indikator] = lang

daten = pd.concat(roh.values(), ignore_index=True).rename(columns={"Gebiet": "gebiet", "Sachmerkmal": "merkmal"})
print(f"{len(daten):,} Zeilen aus {len(roh)} Dateien")
daten.sample(5, random_state=1)

201,428 Zeilen aus 21 Dateien


,indikator,gebiet,merkmal,zeitpunkt,roh
49611,familien,Marienbrunn,Nichteheliche Paare mit 2 Kindern,2017,33
103124,sgb2,Grünau-Mitte,mit Kindern,2025,508
71407,wanderungen,Nordwest,Zuzüge über die Stadtgrenze,2020,2468
110124,arbeitslose,Plaußig-Portitz,mit Berufsausbildung,2004,NaN
112439,arbeitslose,Dölitz-Dösen,ohne Berufsausbildung,2006,NaN


## 4. Prüfen: Welche Werte kommen vor?

Bevor wir umwandeln, prüfen wir, dass jeder Rohwert entweder eine Zahl im deutschen Format oder eines der bekannten Zeichen ist. Taucht etwas Unbekanntes auf, bricht das Notebook hier ab, statt es stillschweigend zu „fehlend“ zu machen.

Vorab geprüft: Die Dateien enthalten **keine Tausenderpunkte**. Ein Punkt kommt nur als Sonderzeichen `.` vor.

In [5]:
ZAHL = r"^-?\d+(,\d+)?$"
ZEICHEN = {
    "-": "nichts vorhanden (0)",
    ".": "unbekannt oder aus Datenschutzgründen gesperrt",
    "x": "Angabe aus sachlichen Gründen nicht möglich",
}

ist_zahl = daten["roh"].str.match(ZAHL, na=False)
ist_zeichen = daten["roh"].isin(ZEICHEN.keys())
ist_leer = daten["roh"].isna()

unbekannt = daten[~(ist_zahl | ist_zeichen | ist_leer)]
assert unbekannt.empty, f"Unbekannte Werte:\n{unbekannt['roh'].value_counts()}"

pd.Series({
    "Zahl": ist_zahl.sum(),
    "Zeichen -": (daten["roh"] == "-").sum(),
    "Zeichen .": (daten["roh"] == ".").sum(),
    "Zeichen x": (daten["roh"] == "x").sum(),
    "leer": ist_leer.sum(),
}, name="Anzahl")

Zahl         172337
Zeichen -       893
Zeichen .      4608
Zeichen x       271
leer          23319
Name: Anzahl, dtype: int64

## 5. Umwandeln: Zeichen und Zahlen

Nach der offiziellen Zeichenerklärung des Amts für Statistik und Wahlen (Entscheidungsprotokoll Nr. 13):

| Zeichen | Bedeutung | wird zu |
|---|---|---|
| `-` | nichts vorhanden | **0** |
| `.` | unbekannt oder Datenschutz | fehlend |
| `x` | Angabe aus sachlichen Gründen nicht möglich | fehlend |
| leer | kein Wert geliefert | fehlend |

Das Originalzeichen bleibt in `zeichen`, die Erklärung in `hinweis`. So kann das Dashboard später zeigen, *warum* ein Wert fehlt.

In [6]:
daten["zeichen"] = daten["roh"].where(ist_zeichen)
daten["hinweis"] = daten["zeichen"].map(ZEICHEN)
daten.loc[ist_leer, "hinweis"] = "kein Wert geliefert"

daten["wert"] = pd.to_numeric(daten["roh"].where(ist_zahl).str.replace(",", "."))
daten.loc[daten["roh"] == "-", "wert"] = 0.0

# Kontrolle: Jede Zahl wurde umgewandelt, jede Nicht-Zahl außer "-" ist fehlend
assert daten.loc[ist_zahl, "wert"].notna().all()
assert daten.loc[~ist_zahl & (daten["roh"] != "-"), "wert"].isna().all()

daten[["roh", "wert", "zeichen", "hinweis"]].drop_duplicates("roh").head(8)

,roh,wert,zeichen,hinweis
0,1896,1896.0,NaN,NaN
1,2546,2546.0,NaN,NaN
2,9245,9245.0,NaN,NaN
3,8331,8331.0,NaN,NaN
4,6594,6594.0,NaN,NaN
5,6209,6209.0,NaN,NaN
6,6165,6165.0,NaN,NaN
7,9370,9370.0,NaN,NaN


## 6. Merkmalsnamen bereinigen

Drei Probleme in den Rohdaten:

1. **Parteinamen:** Bei den Bundestagswahlen heißt es „DIe Linke“, sonst „DIE LINKE“ (Nr. 12). Vereinheitlicht auf „DIE LINKE“.
2. **SGB II:** „Männer“ und „Frauen“ kommen je zweimal vor, einmal unter den *erwerbsfähigen* und einmal unter den *nicht erwerbsfähigen* Leistungsberechtigten. Ohne Präfix wären die Merkmale nicht eindeutig.
3. **Tippfehler:** „25 bis unter 55 ahre“ in SGB II.

In [7]:
UMBENENNEN = {
    ("bundestagswahlen", "Stimmenanteile DIe Linke"): "Stimmenanteile DIE LINKE",
    ("sgb2", "25 bis unter 55 ahre"): "25 bis unter 55 Jahre",
}
for (indikator, alt), neu in UMBENENNEN.items():
    treffer = (daten["indikator"] == indikator) & (daten["merkmal"] == alt)
    assert treffer.any(), f"{indikator}: '{alt}' nicht gefunden"
    daten.loc[treffer, "merkmal"] = neu

In [8]:
# SGB II: In jeder Gebiet-Zeitpunkt-Gruppe stehen die Merkmale in fester Reihenfolge.
# Das erste "Männer"/"Frauen" gehört zu den erwerbsfähigen, das zweite zu den
# nicht erwerbsfähigen Leistungsberechtigten (Reihenfolge in der Rohdatei geprüft).
sgb2 = daten["indikator"] == "sgb2"
nr = daten[sgb2].groupby(["gebiet", "zeitpunkt", "merkmal"]).cumcount()
doppelt = daten.loc[sgb2, "merkmal"].isin(["Männer", "Frauen"])

daten.loc[nr[doppelt & (nr == 0)].index, "merkmal"] = "Erwerbsfähige Leistungsberechtigte: " + daten["merkmal"]
daten.loc[nr[doppelt & (nr == 1)].index, "merkmal"] = "Nichterwerbsfähige: " + daten["merkmal"]

daten.loc[sgb2 & daten["merkmal"].str.contains("Männer|Frauen"), "merkmal"].unique()

<StringArray>
['Erwerbsfähige Leistungsberechtigte: Männer', 'Erwerbsfähige Leistungsberechtigte: Frauen', 'Nichterwerbsfähige: Männer', 'Nichterwerbsfähige: Frauen']
Length: 4, dtype: str

In [9]:
# Jetzt muss jede Kombination aus Indikator, Gebiet, Merkmal und Zeitpunkt eindeutig sein
schluessel = ["indikator", "gebiet", "merkmal", "zeitpunkt"]
assert not daten.duplicated(schluessel).any(), daten[daten.duplicated(schluessel, keep=False)].head()
print("Alle Schlüssel eindeutig.")

Alle Schlüssel eindeutig.


## 7. Zeitpunkte und Gebiete ergänzen

- **Zeitpunkte:** Bei Jahreswerten ist der Zeitpunkt das Jahr, bei Wahlen das Wahldatum. Für beide gibt es zusätzlich eine Spalte `jahr`, damit sich Indikatoren auf einer gemeinsamen Achse vergleichen lassen.
- **Gebiete:** Ebene, Ortsteilnummer und Stadtbezirk aus den Geodaten anhängen. Jedes Gebiet muss einen Partner finden (Nr. 9).

In [10]:
daten = daten.merge(INDIKATOREN[["indikator", "zeitart"]], on="indikator")

ist_wahl = daten["zeitart"] == "wahl"
daten["datum"] = pd.to_datetime(daten["zeitpunkt"].where(ist_wahl), format="%d.%m.%Y")
daten["jahr"] = daten["datum"].dt.year.where(ist_wahl, pd.to_numeric(daten["zeitpunkt"], errors="coerce")).astype("Int64")
assert daten["jahr"].notna().all()

vorher = len(daten)
daten = daten.merge(gebiete, on="gebiet", how="left", validate="many_to_one")
assert len(daten) == vorher
ohne_partner = daten.loc[daten["ebene"].isna(), "gebiet"].unique()
assert len(ohne_partner) == 0, f"Gebiete ohne Partner in den Geodaten: {ohne_partner}"

# Jeder Indikator hat alle 74 Gebiete
gebiete_je_indikator = daten.groupby("indikator")["gebiet"].nunique()
assert (gebiete_je_indikator == len(gebiete)).all(), gebiete_je_indikator[gebiete_je_indikator != len(gebiete)]
print("Alle 21 Indikatoren decken alle 74 Gebiete ab.")

Alle 21 Indikatoren decken alle 74 Gebiete ab.


## 8. Stichproben gegen bekannte Werte

Ein paar Werte, die wir aus den bisherigen Analysen kennen. Stimmen sie, ist die Umwandlung plausibel.

In [11]:
def wert(indikator, gebiet, merkmal, jahr):
    zeile = daten[(daten["indikator"] == indikator) & (daten["gebiet"] == gebiet)
                  & (daten["merkmal"] == merkmal) & (daten["jahr"] == jahr)]
    assert len(zeile) == 1, zeile
    return zeile["wert"].iloc[0]

assert wert("einwohner", "Zentrum", "Einwohner insgesamt", 2024) == 1893         # Hauptwohnsitze (Nr. 2)
assert wert("einwohner", "Heiterblick", "Einwohner insgesamt", 2013) == 3611     # Grenzänderung (Nr. 5)
assert round(wert("bundestagswahlen", "Zentrum", "Stimmenanteile AfD", 2025), 1) == 17.7
assert pd.isna(wert("bundestagswahlen", "Zentrum", "Stimmenanteile AfD", 2009))  # nicht angetreten (Nr. 12)
assert wert("bundestagswahlen", "Zentrum", "Stimmenanteile DIE LINKE", 2025) > 0 # Schreibweise vereinheitlicht
print("Alle Stichproben stimmen.")

Alle Stichproben stimmen.


## 9. Ergebnis ansehen und speichern

In [12]:
SPALTEN = ["indikator", "gebiet", "ebene", "nummer", "stadtbezirk", "merkmal",
           "zeitart", "zeitpunkt", "datum", "jahr", "wert", "zeichen", "hinweis"]
ergebnis = daten[SPALTEN].sort_values(["indikator", "ebene", "nummer", "gebiet", "merkmal", "jahr"]).reset_index(drop=True)

print(f"{len(ergebnis):,} Zeilen, {ergebnis['indikator'].nunique()} Indikatoren, "
      f"{ergebnis['merkmal'].nunique()} Merkmale, {ergebnis['gebiet'].nunique()} Gebiete")
ergebnis.head()

201,428 Zeilen, 21 Indikatoren, 152 Merkmale, 74 Gebiete


,indikator,gebiet,ebene,nummer,stadtbezirk,merkmal,zeitart,zeitpunkt,datum,jahr,wert,zeichen,hinweis
0,arbeitslose,Zentrum,Ortsteil,00,Mitte,Anteil an den Erwerbsfähigen,jahr,2000,NaT,2000,14.9,NaN,NaN
1,arbeitslose,Zentrum,Ortsteil,00,Mitte,Anteil an den Erwerbsfähigen,jahr,2001,NaT,2001,12.1,NaN,NaN
2,arbeitslose,Zentrum,Ortsteil,00,Mitte,Anteil an den Erwerbsfähigen,jahr,2002,NaT,2002,9.6,NaN,NaN
3,arbeitslose,Zentrum,Ortsteil,00,Mitte,Anteil an den Erwerbsfähigen,jahr,2003,NaT,2003,9.0,NaN,NaN
4,arbeitslose,Zentrum,Ortsteil,00,Mitte,Anteil an den Erwerbsfähigen,jahr,2004,NaT,2004,9.3,NaN,NaN


In [13]:
# Überblick je Indikator: Zeitraum, Anzahl Merkmale, Anteil fehlender Werte bei den Ortsteilen
ot = ergebnis[ergebnis["ebene"] == "Ortsteil"]
ueberblick = ot.groupby("indikator").agg(
    von=("jahr", "min"),
    bis=("jahr", "max"),
    zeitpunkte=("zeitpunkt", "nunique"),
    merkmale=("merkmal", "nunique"),
    fehlend=("wert", lambda w: f"{w.isna().mean():.1%}"),
)
INDIKATOREN.merge(ueberblick, left_on="indikator", right_index=True)

,indikator,titel,thema,zeitart,umfrage,von,bis,zeitpunkte,merkmale,fehlend
0,einwohner,Einwohner (Hauptwohnsitz),Bevölkerung,jahr,False,2000,2025,26,5,0.0%
1,einwohner_alter,Einwohner nach Alter,Bevölkerung,jahr,False,2000,2025,26,3,0.0%
2,migrationshintergrund,Einwohner mit Migrationshintergrund,Bevölkerung,jahr,False,2008,2025,18,8,0.2%
3,einwohnerdichte,Einwohnerdichte,Bevölkerung,jahr,False,2004,2025,22,1,0.0%
4,haushalte,Personenhaushalte,Bevölkerung,jahr,False,2006,2025,20,12,0.0%
5,familien,Familien mit Kindern,Bevölkerung,jahr,False,2015,2025,11,22,5.4%
6,geborene_gestorbene,Geborene und Gestorbene,Bevölkerung,jahr,False,2000,2025,26,3,0.0%
7,wanderungen,Wanderungen,Bevölkerung,jahr,False,2015,2025,11,6,0.1%
8,sgb2,Grundsicherung für Arbeitssuchende (SGB II),Soziale Lage,jahr,False,2008,2025,18,23,1.8%
9,arbeitslose,Arbeitslose,Soziale Lage,jahr,False,2000,2025,26,16,37.1%


**Hohe Anteile fehlender Werte sind meist keine Lücken.** Viele Merkmale wurden erst später eingeführt und sind davor leer (Hinweis „kein Wert geliefert“):

- Arbeitslose nach Schul- und Berufsabschluss: erst ab 2018
- Arbeitslose nach SGB II / SGB III: erst ab 2006
- Kfz nach Antrieb und Schadstoffklasse: erst ab 2015–2017
- Straftaten: Vermögensdelikte erst ab 2010
- Flächennutzung: 2016 neue Kategorien. „Gebäude- und Freifläche“ (bis 2014) wurde durch „Wohnbaufläche“ und „Industrie- und Gewerbefläche“ ersetzt.

Echte Lücken durch Datenschutz (`.`) betreffen vor allem kleine Gruppen und die Umfragedaten (Mieten 15–19 %).

In [14]:
# Wann beginnt jedes Merkmal? Merkmale, die später starten als ihr Indikator
start = ot[ot["wert"].notna()].groupby(["indikator", "merkmal"])["jahr"].min().rename("erster_wert").reset_index()
start = start.merge(ot.groupby("indikator")["jahr"].min().rename("indikator_ab"), on="indikator")
start[start["erster_wert"] > start["indikator_ab"]].sort_values(["indikator", "erster_wert"])

,indikator,merkmal,erster_wert,indikator_ab
10,arbeitslose,nach SGB II,2006,2000
11,arbeitslose,nach SGB III,2006,2000
5,arbeitslose,mit Abitur,2018,2000
6,arbeitslose,mit Berufsausbildung,2018,2000
7,arbeitslose,mit Hauptschulabschluss,2018,2000
8,arbeitslose,mit Mittlerer Reife,2018,2000
9,arbeitslose,mit akademischer Ausbildung,2018,2000
12,arbeitslose,ohne Berufsausbildung,2018,2000
13,arbeitslose,ohne Schulabschluss,2018,2000
16,bundestagswahlen,Stimmenanteile AfD,2013,1994


In [15]:
ergebnis.to_csv(PROCESSED / "kleinraeumig_lang.csv", index=False, encoding="utf-8")
INDIKATOREN.to_csv(PROCESSED / "indikatoren.csv", index=False, encoding="utf-8")
print("Gespeichert:", *sorted(p.name for p in PROCESSED.glob("*.csv")))

Gespeichert: indikatoren.csv kleinraeumig_lang.csv


## So liest man die Daten später ein

```python
daten = pd.read_csv("data/processed/kleinraeumig_lang.csv", dtype={"nummer": str, "zeitpunkt": str}, parse_dates=["datum"])
```

`dtype={...}` ist wichtig: Sonst wird aus der Ortsteilnummer `"05"` die Zahl `5`, und die Spalte `zeitpunkt` mischt Jahreszahlen mit Wahldaten.

## Noch nicht enthalten

- **Bürgerumfrage 2024:** anders aufgebaut (Fragen × Antworten × Bevölkerungsgruppen), nur nach Stadtbezirken gegliedert. Wird bei Bedarf separat aufbereitet.
- **Markierung der Grenzänderungen** (2006, 2012, 2014, Nr. 5) und **Glättung der Umfragedaten** (Nr. 14): Das sind Darstellungsfragen und gehören ins Dashboard, nicht in die Daten.